In [1]:
from pathlib import Path
from IPython.display import display, Markdown

from resources.imports import *
import torch

from resources.MLmetrics import (
    postprocess_resolve_hpo_run_path,
    postprocess_hpo_path_summary,
    postprocess_list_hpo_runs,
    postprocess_cross_hpo_base_path,
    postprocess_load_hpo_study,
    postprocess_hpo_best_overview,
    display_hpo_best_overview,
    postprocess_hpo_trial_leaderboard,
    plot_hpo_optimization_history,
    postprocess_hpo_param_importance,
    plot_hpo_param_importance,
    postprocess_full_hpo_comparison,
    postprocess_select_full_hpo_model,
    plot_full_hpo_comparison,
    postprocess_artifact_table,
    postprocess_load_curve_run,
    postprocess_curve_run_overview,
    postprocess_build_active_curve_diagnostics,
    curve_summary_table,
    print_curve_diagnostics,
    display_curve_main_dashboard,
    display_curve_sample_examples,
    postprocess_load_field_run,
    postprocess_field_run_overview,
    postprocess_build_active_field_diagnostics,
    field_summary_table,
    print_field_diagnostics,
    plot_field_frame_component_trends,
    plot_field_diversity,
    display_field_sample_error_summary,
    plot_loss_history,
)

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ImportError: cannot import name 'postprocess_full_hpo_comparison' from 'resources.MLmetrics' (C:\Users\exy053\Documents\00-PhD-gitRepo\resources\MLmetrics.py)

In [ ]:
%load_ext autoreload
%autoreload 2

# ML HPO Post-Processing

## 1. Find Recent HPO Runs

In [ ]:
FIND_RUN = False
RUN_ROOT = Path(r"Z:/p2")
MAX_RECENT_HPO_RUNS = 25


if FIND_RUN:
    recent_hpo_runs = postprocess_list_hpo_runs(RUN_ROOT, max_runs=MAX_RECENT_HPO_RUNS)
    display(recent_hpo_runs)
    if recent_hpo_runs.empty:
        raise ValueError("No HPO studies with full_study.db were found. Set RUN_PATH_OVERRIDE or update RUN_ROOT.")

## 2. User Configuration

In [ ]:
OUTPUT_KIND = "curve"  # "curve" or "field"
RUN_TYPE = "full_hpo"  # "model_hpo" or "full_hpo"
TASK = "UT"
mechMode = TASK
MODEL = "auto"  # "auto" selects the best family; or set MLP, GCN, GAT, Transformer.
RUN_NAME = "fFT-fHPO"
run_name = RUN_NAME
RUN_PATH_OVERRIDE = None
STUDY_NAME = None

DATA_PATH_OVERRIDE = None
PREFER_HPO_BEST = True
LOAD_DATA = True
LOAD_MODEL = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ACTIVE_SPLIT = None
VIEW_MODE = None
ZONE_BOUNDARIES = None

CROSS_MODEL_FAMILIES = None  # None discovers folders such as MLP, GCN, GAT, and Transformer.
CROSS_MODEL_SELECTION_METRIC = "best_value"

OUTPUT_KIND = str(OUTPUT_KIND).lower()
if OUTPUT_KIND not in ["curve", "field"]:
    raise ValueError("OUTPUT_KIND must be 'curve' or 'field'.")

RUN_TYPE = str(RUN_TYPE).lower()
if RUN_TYPE not in ["model_hpo", "full_hpo"]:
    raise ValueError("RUN_TYPE must be 'model_hpo' or 'full_hpo'.")

if VIEW_MODE is None:
    VIEW_MODE = TASK if str(TASK).upper() in ["UT", "FT"] else "UT"
VIEW_MODE = str(VIEW_MODE).upper()
if VIEW_MODE not in ["UT", "FT"]:
    raise ValueError("VIEW_MODE must be UT or FT. For MULTI runs, choose which output branch to view.")

cross_model_comparison = None
selected_model_metric = None
selected_model_row = None
requested_model = str(MODEL).strip()

if RUN_TYPE == "full_hpo":
    output_token = {"curve": "Curve", "field": "Field"}[OUTPUT_KIND]
    CROSS_HPO_BASE = Path(RUN_ROOT).expanduser() / str(TASK).upper() / output_token / "HPO" / str(RUN_NAME)
    cross_model_comparison = postprocess_full_hpo_comparison(
        CROSS_HPO_BASE,
        model_names=CROSS_MODEL_FAMILIES,
        task=VIEW_MODE,
    )
    if requested_model.lower() in ["auto", "best", "overall"]:
        selected_model, selected_model_metric, selected_model_row = postprocess_select_full_hpo_model(
            cross_model_comparison,
            preferred_metric=CROSS_MODEL_SELECTION_METRIC,
            task=VIEW_MODE,
        )
        if selected_model is None:
            raise ValueError(f"Could not select a model from cross-model HPO results under {CROSS_HPO_BASE}.")
        MODEL = selected_model
        print(f"Selected MODEL={MODEL} using {selected_model_metric} from the cross-model HPO comparison.")

model = MODEL
RUN_PATH = postprocess_resolve_hpo_run_path(
    run_root=RUN_ROOT,
    task=TASK,
    output_kind=OUTPUT_KIND,
    model=MODEL,
    run_name=RUN_NAME,
    run_type=RUN_TYPE,
    run_path_override=RUN_PATH_OVERRIDE,
)
CROSS_HPO_BASE = postprocess_cross_hpo_base_path(RUN_PATH, run_type=RUN_TYPE)
if RUN_TYPE != "full_hpo":
    CROSS_HPO_BASE = None


NameError: name 'torch' is not defined

## 3. Run And Artifact Discovery

In [ ]:
ARTIFACT_TABLE_KEYS = None


display(postprocess_hpo_path_summary(
    RUN_PATH,
    run_root=RUN_ROOT,
    task=TASK,
    output_kind=OUTPUT_KIND,
    model=MODEL,
    run_name=RUN_NAME,
    run_type=RUN_TYPE,
))

if OUTPUT_KIND == "curve":
    artifacts, loaded, DAT, MOD = postprocess_load_curve_run(
        RUN_PATH,
        run_root=RUN_ROOT,
        prefer_hpo_best=PREFER_HPO_BEST,
        load_data=LOAD_DATA,
        load_model=LOAD_MODEL,
        data_path_override=DATA_PATH_OVERRIDE,
        device=DEVICE,
    )
elif OUTPUT_KIND == "field":
    artifacts, loaded, DAT, MOD = postprocess_load_field_run(
        RUN_PATH,
        run_root=RUN_ROOT,
        prefer_hpo_best=PREFER_HPO_BEST,
        load_data=LOAD_DATA,
        load_model=LOAD_MODEL,
        data_path_override=DATA_PATH_OVERRIDE,
        device=DEVICE,
    )


display(postprocess_artifact_table(artifacts, keys=ARTIFACT_TABLE_KEYS))
for warning in artifacts.get("warnings", []):
    print("WARNING:", warning)

## 4. Load Study

In [ ]:
study_info = postprocess_load_hpo_study(RUN_PATH, study_name=STUDY_NAME)
study = study_info["study"]

print(study_info["message"])
if not study_info["summaries"].empty:
    display(study_info["summaries"])
if study is None:
    print("Study-dependent HPO summaries will stay empty until full_study.db can be loaded.")

## 5. Best Trial Overview

In [ ]:
MAX_BEST_USER_ATTRS = 20


best_overview = postprocess_hpo_best_overview(study_info, artifacts=artifacts, loaded=loaded)
display_hpo_best_overview(best_overview, max_attrs=MAX_BEST_USER_ATTRS)

## 6. Trial Leaderboard

In [ ]:
LEADERBOARD_TOP_N = 20
LEADERBOARD_KEY_PARAMS = None


leaderboard = postprocess_hpo_trial_leaderboard(
    study,
    top_n=LEADERBOARD_TOP_N,
    key_params=LEADERBOARD_KEY_PARAMS,
)
if leaderboard.empty:
    print("No trials are available for the leaderboard.")
else:
    display(leaderboard)

## 7. Optimization History

In [ ]:
OPT_HISTORY_FIGSIZE = (9, 4)


plot_hpo_optimization_history(study, figsize=OPT_HISTORY_FIGSIZE)

## 8. Parameter Importance

In [ ]:
IMPORTANCE_TOP_N = 15
IMPORTANCE_FIGSIZE = None


importance, importance_message = postprocess_hpo_param_importance(study)
if importance_message:
    print(importance_message)
else:
    display(importance)
    plot_hpo_param_importance(importance, top_n=IMPORTANCE_TOP_N, figsize=IMPORTANCE_FIGSIZE)

## 9. Cross-Model Comparison

In [ ]:
RUN_CROSS_MODEL_COMPARISON = RUN_TYPE == "full_hpo"
CROSS_COMPARE_FIGSIZE = (9, 4)


if RUN_CROSS_MODEL_COMPARISON:
    if cross_model_comparison is None:
        cross_model_comparison = postprocess_full_hpo_comparison(
            CROSS_HPO_BASE,
            model_names=CROSS_MODEL_FAMILIES,
            task=VIEW_MODE,
        )
    compact_columns = [
        "model",
        "best_value",
        "best_trial",
        "n_trials",
        "n_complete",
        "n_pruned",
        "n_failed",
        "evaluation_split",
        f"{VIEW_MODE}_best_rmse",
        f"{VIEW_MODE}_val_rmse",
        f"{VIEW_MODE}_summary_rmse",
        f"{VIEW_MODE}_summary_skill_vs_mean_curve_rmse",
        f"{VIEW_MODE}_summary_skill_vs_mean_field_rmse",
        "best_model_json",
    ]
    compact_columns = [col for col in compact_columns if col in cross_model_comparison.columns]
    if cross_model_comparison.empty:
        print("No cross-model HPO folders were found under:", CROSS_HPO_BASE)
    else:
        display(cross_model_comparison[compact_columns])
        plot_full_hpo_comparison(cross_model_comparison, figsize=CROSS_COMPARE_FIGSIZE)
        print(f"Detailed best-model diagnostics are loaded from MODEL={MODEL}.")
        if selected_model_metric is not None:
            print(f"Model selection metric: {selected_model_metric}")
else:
    print("Cross-model comparison is skipped because RUN_TYPE is not 'full_hpo'.")

## 10. Best-Model Run Setup

In [ ]:
SHOW_BEST_MODEL_RUN_SETUP = True


if OUTPUT_KIND == "curve":
    best_run_overview = postprocess_curve_run_overview(
        artifacts,
        loaded,
        data=DAT,
        run_name=RUN_NAME,
        run_type=RUN_TYPE,
        mech_mode=TASK,
        view_mode=VIEW_MODE,
        model_name=MODEL,
        device=DEVICE,
        active_split=ACTIVE_SPLIT,
    )
    available_evals = best_run_overview["available_evals"]
    available_output_evals = best_run_overview["available_curve_evals"]
elif OUTPUT_KIND == "field":
    best_run_overview = postprocess_field_run_overview(
        artifacts,
        loaded,
        data=DAT,
        run_name=RUN_NAME,
        run_type=RUN_TYPE,
        mech_mode=TASK,
        view_mode=VIEW_MODE,
        model_name=MODEL,
        device=DEVICE,
        active_split=ACTIVE_SPLIT,
    )
    available_evals = best_run_overview["available_evals"]
    available_output_evals = best_run_overview["available_field_evals"]

ACTIVE_SPLIT = best_run_overview["active_split"]

if SHOW_BEST_MODEL_RUN_SETUP:
    display(best_run_overview["summary"])
    display(Markdown("### Available Saved Evaluations"))
    display(available_output_evals)
print("OUTPUT_KIND:", OUTPUT_KIND)
print("VIEW_MODE:", VIEW_MODE)
print("ACTIVE_SPLIT:", ACTIVE_SPLIT)

## 11. Build Best-Model Diagnostics

In [ ]:
PRINT_BEST_MODEL_DIAGNOSTICS = True


if OUTPUT_KIND == "curve":
    diagnostics, active_key, active_diag = postprocess_build_active_curve_diagnostics(
        DAT,
        loaded,
        available_evals,
        view_mode=VIEW_MODE,
        active_split=ACTIVE_SPLIT,
        model=MOD,
        zone_boundaries=ZONE_BOUNDARIES,
    )
elif OUTPUT_KIND == "field":
    diagnostics, active_key, active_diag = postprocess_build_active_field_diagnostics(
        DAT,
        loaded,
        available_evals,
        view_mode=VIEW_MODE,
        active_split=ACTIVE_SPLIT,
        model=MOD,
    )

if active_diag is None:
    print("No active best-model diagnostics are available. Check predictions.npz, diagnostic CSVs, VIEW_MODE, or ACTIVE_SPLIT.")
elif PRINT_BEST_MODEL_DIAGNOSTICS and OUTPUT_KIND == "curve":
    print_curve_diagnostics(active_diag, label=f"{active_key[0]} {active_key[1]}")
elif PRINT_BEST_MODEL_DIAGNOSTICS and OUTPUT_KIND == "field":
    print_field_diagnostics(active_diag, label=f"{active_key[0]} {active_key[1]}")

## 12. Best-Model Diagnostic Summary

In [ ]:
CURVE_SUMMARY_METRICS = [
    "rmse",
    "mae",
    "mse",
    "bias",
    "r2_global",
    "collapse_ratio",
    "mean_curve_baseline_rmse",
    "skill_vs_mean_curve_rmse",
    "mean_sample_curve_corr",
    "peak_corr",
    "energy_corr",
    "n_samples",
    "n_points",
]
FIELD_SUMMARY_METRICS = [
    "rmse",
    "mae",
    "mse",
    "bias",
    "collapse_ratio",
    "mean_field_baseline_rmse",
    "skill_vs_mean_field_rmse",
    "valid_fraction",
    "n_samples",
    "n_frames",
    "n_nodes",
    "n_components",
]


if active_diag is None:
    print("No active best-model diagnostics are available.")
elif OUTPUT_KIND == "curve":
    display(curve_summary_table(active_diag, metrics=CURVE_SUMMARY_METRICS))
elif OUTPUT_KIND == "field":
    display(field_summary_table(active_diag, metrics=FIELD_SUMMARY_METRICS))

## 13. Light Best-Model Visual Check

In [ ]:
CURVE_DASHBOARD_MAX_SAMPLES = 32
CURVE_DASHBOARD_SORT_BY = "rmse"
SHOW_CURVE_SAMPLE_EXAMPLES = True
CURVE_SELECTED_SAMPLE = 0
CURVE_RANDOM_SAMPLE_COUNT = 4
CURVE_RANDOM_SEED = 42

FIELD_FRAME_TRENDS_FIGSIZE = (14, 8)
FIELD_SHOW_DIVERSITY = True
FIELD_DIVERSITY_FIGSIZE = (16, 4)
FIELD_SAMPLE_TABLE_TOP_N = 3
FIELD_SAMPLE_DISTRIBUTION_BINS = 30


if active_diag is None:
    print("No active best-model diagnostics are available.")
elif OUTPUT_KIND == "curve":
    display_curve_main_dashboard(
        active_diag,
        data=DAT,
        mode=active_key[0] if active_key is not None else VIEW_MODE,
        max_samples=CURVE_DASHBOARD_MAX_SAMPLES,
        sort_by=CURVE_DASHBOARD_SORT_BY,
    )
    if SHOW_CURVE_SAMPLE_EXAMPLES:
        display_curve_sample_examples(
            active_diag,
            selected_sample=CURVE_SELECTED_SAMPLE,
            random_count=CURVE_RANDOM_SAMPLE_COUNT,
            random_seed=CURVE_RANDOM_SEED,
            rank_by="sample_rmse",
            ncols=2,
        )
elif OUTPUT_KIND == "field":
    plot_field_frame_component_trends(active_diag, figsize=FIELD_FRAME_TRENDS_FIGSIZE)
    if FIELD_SHOW_DIVERSITY:
        try:
            plot_field_diversity(active_diag, figsize=FIELD_DIVERSITY_FIGSIZE)
        except ValueError as exc:
            print(exc)
    display_field_sample_error_summary(
        active_diag,
        bins=FIELD_SAMPLE_DISTRIBUTION_BINS,
        ncols=3,
        top_n=FIELD_SAMPLE_TABLE_TOP_N,
        columns=["sample_mae", "sample_mse", "sample_rmse", "sample_bias", "valid_fraction"],
    )

## 14. Best-Model Loss History

In [ ]:
LOSS_HISTORY_METRICS = ["train_loss", "val_loss"]
LOSS_HISTORY_FIGSIZE = (9, 4)


loss_history = loaded.get("loss_history")
if loss_history is not None and hasattr(loss_history, "empty") and not loss_history.empty:
    display(loss_history.head())
plot_loss_history(loss_history, metrics=LOSS_HISTORY_METRICS, figsize=LOSS_HISTORY_FIGSIZE)